# 🔍 InFi-Check Demo — Gradio trên Colab

**Kiểm chứng tính chính xác của tóm tắt do LLM sinh ra**

| | |
|---|---|
| Model | Qwen2.5-7B-Instruct + LoRA adapter |
| UI | Gradio (public link tự động) |
| Flow | Document → **Sinh tóm tắt** → Chỉnh sửa → **Kiểm tra** |

---
**Cách dùng:**
1. Điền đường dẫn model vào `MODEL_PATH` ở Bước 2 (sau khi fine-tune xong)
2. Đặt `USE_MOCK = True` nếu chưa có model (test UI)
3. Run All → đợi link Gradio xuất hiện → Click để mở giao diện

## Bước 1: Cài đặt thư viện

In [1]:
!pip install -q gradio transformers peft accelerate bitsandbytes sentencepiece groq
print('✅ Cài đặt xong!')

✅ Cài đặt xong!


## Bước 2: Cấu hình

In [2]:
from google.colab import drive
# Thêm force_remount=True để ép buộc xác thực lại nếu muốn đổi tài khoản
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
from google.colab import userdata

LORA_ADAPTER_PATH = "/content/drive/MyDrive/infi-check-model"
BASE_MODEL        = 'Qwen/Qwen2.5-7B-Instruct'
USE_MOCK          = False

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = ''
    print('⚠️  OPENAI_API_KEY chưa được set trong Colab Secrets')

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = ''
    print('⚠️  GROQ_API_KEY chưa được set trong Colab Secrets')

GPT_MODEL  = 'gpt-4o-mini'
GROQ_MODEL = 'llama-3.3-70b-versatile'

MAX_INPUT_TOKENS = 3072

print(f'Chế độ: {"⚠️  MOCK (test UI)" if USE_MOCK else "✅ Model thật: " + LORA_ADAPTER_PATH}')
print(f'OpenAI key: {"✅" if OPENAI_API_KEY else "❌ chưa set"}')
print(f'Groq key  : {"✅" if GROQ_API_KEY else "❌ chưa set"}')
print(f'Max input tokens: {MAX_INPUT_TOKENS}')

Chế độ: ✅ Model thật: /content/drive/MyDrive/infi-check-model
OpenAI key: ✅
Groq key  : ✅
Max input tokens: 3072


## Bước 2b: Khởi tạo API clients (GPT-4o-mini & Groq)*



In [4]:
from openai import OpenAI
from groq import Groq

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
groq_client   = Groq(api_key=GROQ_API_KEY)     if GROQ_API_KEY   else None

print(f'OpenAI client: {"✅" if openai_client else "❌"}')
print(f'Groq client  : {"✅" if groq_client else "❌"}')


OpenAI client: ✅
Groq client  : ✅


## Bước 3: Load model

In [5]:
import torch, re, time

model = None
tokenizer = None

# Hàm clean document dùng lại từ training pipeline
def clean_doc_for_inference(text: str) -> str:
    cutoff_markers = [
        'Tặng sao', 'Chuyển sao', 'Tuổi Trẻ Online', '© Copyright',
        'Tin cùng chuyên mục', 'Tuổi Trẻ Sao', 'Thông tin tài khoản',
        'Hotline:', 'Địa chỉ:', 'Tổng biên tập:',
        'Vui lòng nhập Email', 'Hiện chưa có bình luận',
    ]
    for m in cutoff_markers:
        if m in text:
            text = text[:text.index(m)]
    text = re.sub(r'&[a-z]+_[^\s]+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

if not USE_MOCK:
    print('⏳ Load Base Model (4-bit) + LoRA Adapter...')
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    USE_BF16 = torch.cuda.is_bf16_supported()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )

    # ✅ Load LoRA adapter từ đường dẫn đúng
    model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
    model.eval()

    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ Model loaded!')
    print(f'   VRAM: {vram_used:.1f} / {vram_total:.1f} GB  ({vram_used/vram_total*100:.0f}%)')
    print(f'   Adapter: r=16, alpha=32 | Base: Qwen2.5-7B-Instruct')
    print(f'   Train loss: 5.34 | Eval loss: 0.340 (step 120)')
else:
    print('ℹ️  Chế độ Mock — bỏ qua load model')

⏳ Load Base Model (4-bit) + LoRA Adapter...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model loaded!
   VRAM: 5.7 / 15.6 GB  (37%)
   Adapter: r=16, alpha=32 | Base: Qwen2.5-7B-Instruct
   Train loss: 5.34 | Eval loss: 0.340 (step 120)


## Bước 4: Định nghĩa logic

In [12]:
import gradio as gr
import re, json as _json, time

# ──────────────────────────────────────────────────────────────────────
# HẰNG SỐ MÀU SẮC & NHÃN TIẾNG VIỆT (phải định nghĩa trước)
# ──────────────────────────────────────────────────────────────────────

ERROR_COLORS = {
    "Predicate Error":      {"color": "#EF4444", "bg": "#FEF2F2", "emoji": "🔴"},
    "Entity Error":         {"color": "#F59E0B", "bg": "#FFFBEB", "emoji": "🟡"},
    "Circumstantial Error": {"color": "#F97316", "bg": "#FFF7ED", "emoji": "🟠"},
    "Coreference Error":    {"color": "#6366F1", "bg": "#EEF2FF", "emoji": "🟣"},
    "Discourse Link Error": {"color": "#EC4899", "bg": "#FDF2F8", "emoji": "🔵"},
    "Extrinsic Error":      {"color": "#22C55E", "bg": "#F0FDF4", "emoji": "🟢"},
}

ETYPE_VI = {
    "Predicate Error":      "Lỗi vị ngữ",
    "Entity Error":         "Lỗi thực thể",
    "Circumstantial Error": "Lỗi hoàn cảnh",
    "Coreference Error":    "Lỗi đồng tham chiếu",
    "Discourse Link Error": "Lỗi liên kết diễn ngôn",
    "Extrinsic Error":      "Lỗi ngoại lai",
}

# ──────────────────────────────────────────────────────────────────────
# HELPER RENDER
# ──────────────────────────────────────────────────────────────────────

def _info_box(icon, color, msg):
    return (
        f'<div style="background:{color}15;border:1px solid {color};border-radius:10px;'
        f'padding:14px 18px;color:{color};font-weight:600;">{icon} {msg}</div>'
    )


def _highlight_summary(summary: str, errors: list) -> str:
    import difflib, html as _html_mod
    sentences = re.split(r"(?<=[.!?])\s+", summary.strip())
    parts = []
    for sent in sentences:
        matched_errors = []
        sent_lower = sent.lower().strip()
        for err in errors:
            erroneous = err.get("erroneous_sentence", "").lower().strip()
            if not erroneous:
                continue
            if erroneous[:50] in sent_lower or sent_lower[:50] in erroneous:
                matched_errors.append(err); continue
            ratio = difflib.SequenceMatcher(None, sent_lower, erroneous).ratio()
            if ratio >= 0.55:
                matched_errors.append(err); continue
            specific = err.get("specific", "").lower().strip()
            if specific and len(specific) >= 3 and specific in sent_lower:
                matched_errors.append(err)
        if matched_errors:
            display = sent
            for matched in matched_errors:
                etype    = matched.get("error_type", "Extrinsic Error")
                info     = ERROR_COLORS.get(etype, ERROR_COLORS["Extrinsic Error"])
                specific = matched.get("specific", "")
                if specific and specific in display:
                    escaped = _html_mod.escape(specific)
                    display = display.replace(
                        specific,
                        f'<mark style="background:{info["color"]}33;color:{info["color"]};'
                        f'font-weight:700;padding:0 2px;border-radius:3px;">{escaped}</mark>'
                        f'<sup style="font-size:10px;margin-left:2px;">{info["emoji"]}</sup>'
                    )
            first_info = ERROR_COLORS.get(
                matched_errors[0].get("error_type", "Extrinsic Error"),
                ERROR_COLORS["Extrinsic Error"]
            )
            parts.append(
                f'<span style="background:{first_info["bg"]};border-bottom:2px solid {first_info["color"]};'
                f'border-radius:4px;padding:1px 4px;">{display}</span>'
            )
        else:
            parts.append(f'<span>{sent}</span>')
    return " ".join(parts)


def _render_errors(errors: list, summary: str) -> str:
    if not errors:
        return _info_box("ℹ️", "#6B7280", "Không phân tích được chi tiết lỗi. Xem tab Đầu ra thô.")
    highlighted = _highlight_summary(summary, errors)
    html = f"""
    <div style="margin-bottom:20px;">
        <div style="font-size:11px;font-weight:600;color:#6B7280;text-transform:uppercase;
                    letter-spacing:0.8px;margin-bottom:8px;">📝 BẢN TÓM TẮT (câu chứa lỗi được tô màu)</div>
        <div style="background:#FAFAFA;border:1px solid #E5E7EB;border-radius:10px;
                    padding:16px 18px;font-size:15px;line-height:2.4;color:#1F2937;">
            {highlighted}
        </div>
    </div>
    <div style="font-size:11px;font-weight:600;color:#6B7280;text-transform:uppercase;
                letter-spacing:0.8px;margin-bottom:10px;">🔍 CHI TIẾT LỖI THỰC TẾ</div>
    """
    for err in errors:
        etype    = err.get("error_type", "Extrinsic Error")
        etype_vi = ETYPE_VI.get(etype, etype)
        info     = ERROR_COLORS.get(etype, ERROR_COLORS["Extrinsic Error"])
        specific = err.get("specific", "")
        specific_html = (
            f'<span style="background:{info["color"]}22;color:{info["color"]};font-weight:700;'
            f'padding:0 4px;border-radius:3px;margin-left:4px;">{specific}</span>'
            if specific else ""
        )
        html += f"""
        <div style="background:{info['bg']};border-left:4px solid {info['color']};
                    border-radius:10px;padding:16px 18px;margin-bottom:14px;">
            <span style="display:inline-block;background:{info['color']};color:white;
                         font-size:11px;font-weight:700;padding:3px 12px;border-radius:20px;
                         margin-bottom:12px;">{info['emoji']} {etype_vi}</span>
            <div style="font-size:10px;font-weight:700;color:#9CA3AF;text-transform:uppercase;
                        letter-spacing:0.6px;margin-bottom:3px;">Câu chứa lỗi</div>
            <div style="font-size:14px;color:#374151;margin-bottom:12px;line-height:1.7;">
                {__import__('html').escape(str(err.get('erroneous_sentence', '—')))}{specific_html}
            </div>
            <div style="font-size:10px;font-weight:700;color:#9CA3AF;text-transform:uppercase;
                        letter-spacing:0.6px;margin-bottom:3px;">Giải thích</div>
            <div style="font-size:14px;color:#374151;margin-bottom:12px;line-height:1.7;">
                {err.get('explanation', '—')}
            </div>
            <div style="background:#F0FDF4;border:1px solid #86EFAC;border-radius:8px;
                        padding:11px 14px;font-size:13px;color:#166534;line-height:1.6;">
                ✏️ <b>Bản sửa:</b> {err.get('correction', '—')}
            </div>
        </div>"""
    return html


def _render_references(refs: list) -> str:
    if not refs:
        return _info_box("✅", "#22C55E", "Bản tóm tắt trung thực. Không phát hiện lỗi thực tế.")
    html = (
        '<div style="font-size:11px;font-weight:600;color:#6B7280;text-transform:uppercase;'
        'letter-spacing:0.8px;margin-bottom:10px;">📌 CÂU NGUỒN HỖ TRỢ TỪ VĂN BẢN GỐC</div>'
    )
    for ref in refs:
        sents_html = "".join([
            f'<div style="font-size:13px;color:#374151;line-height:1.7;font-style:italic;'
            f'border-left:2px solid #93C5FD;padding-left:10px;margin-top:6px;">&#8220;{s}&#8221;</div>'
            for s in ref["supporting"]
        ])
        html += f"""
        <div style="background:#F8FAFC;border-left:3px solid #3B82F6;border-radius:8px;
                    padding:14px 16px;margin-bottom:10px;">
            <div style="font-size:12px;font-weight:700;color:#2563EB;margin-bottom:4px;">
                📎 Câu tóm tắt số {ref['sentence_num']}
            </div>
            {sents_html}
        </div>"""
    return html


# ──────────────────────────────────────────────────────────────────────
# HÀM LOGIC CHÍNH: generate_summary, iterative_factcheck,
#                  run_comparison, _translate_raw_vi
# ──────────────────────────────────────────────────────────────────────

def _call_groq(prompt: str, system: str = "") -> str:
    if not groq_client:
        return "[LỖI] Groq client chưa được khởi tạo."
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL, messages=msgs, temperature=0.2, max_tokens=1024
    )
    return resp.choices[0].message.content.strip()


def _call_openai(prompt: str, system: str = "") -> str:
    if not openai_client:
        return "[LỖI] OpenAI client chưa được khởi tạo."
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = openai_client.chat.completions.create(
        model=GPT_MODEL, messages=msgs, temperature=0.2, max_tokens=1024
    )
    return resp.choices[0].message.content.strip()


def _generate_with_model(document: str) -> str:
    """Sinh tóm tắt bằng LoRA model (nếu có) hoặc Groq."""
    if model is not None and tokenizer is not None:
        prompt = (
            f"Hãy tóm tắt văn bản sau bằng tiếng Việt, ngắn gọn và trung thực:\n\n{document}"
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=MAX_INPUT_TOKENS).to(model.device)
        with __import__("torch").no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        # Bỏ phần prompt lặp lại ở đầu output
        if decoded.startswith(prompt):
            decoded = decoded[len(prompt):].strip()
        return decoded
    # Fallback: dùng Groq
    return _call_groq(
        f"Tóm tắt văn bản sau bằng tiếng Việt, 3-5 câu, trung thực:\n\n{document}",
        system="Bạn là trợ lý tóm tắt tin tức tiếng Việt."
    )


def generate_summary(document: str) -> str:
    if USE_MOCK:
        time.sleep(0.5)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", document.strip()) if s.strip()]
        return " ".join(sents[:3]) if sents else "[LỖI] Văn bản trống."
    try:
        return _generate_with_model(document)
    except Exception as e:
        return f"[LỖI] Không sinh được tóm tắt: {e}"


def _parse_factcheck_response(raw: str) -> dict:
    """Phân tích JSON trả về từ LLM fact-checker."""
    try:
        clean = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
        data  = _json.loads(clean)
        verdict = str(data.get("verdict", "")).strip().upper()
        if verdict not in ("YES", "NO"):
            verdict = "NO"
        return {
            "verdict":    verdict,
            "errors":     data.get("errors", []),
            "references": data.get("references", []),
            "raw":        raw,
        }
    except Exception:
        # Nếu không parse được JSON, dùng heuristic
        upper = raw.upper()
        verdict = "YES" if "\"VERDICT\": \"YES\"" in upper or "'VERDICT': 'YES'" in upper else "NO"
        return {"verdict": verdict, "errors": [], "references": [], "raw": raw}


FACTCHECK_SYSTEM = """Bạn là hệ thống kiểm chứng thông tin. Nhiệm vụ: xác định các câu trong bản tóm tắt có chứa thông tin SAI LỆCH so với văn bản gốc không.

Trả về JSON theo đúng định dạng sau (không thêm gì khác):
{
  "verdict": "YES" hoặc "NO",
  "errors": [
    {
      "erroneous_sentence": "câu chứa lỗi trong tóm tắt",
      "error_type": "Predicate Error | Entity Error | Circumstantial Error | Coreference Error | Discourse Link Error | Extrinsic Error",
      "specific": "cụm từ sai cụ thể",
      "explanation": "giải thích lỗi",
      "correction": "bản sửa đúng"
    }
  ],
  "references": [
    {
      "sentence_num": 1,
      "supporting": ["câu nguồn hỗ trợ từ văn bản gốc"]
    }
  ]
}

verdict = "YES" nếu có ít nhất 1 lỗi thực tế. verdict = "NO" nếu tóm tắt trung thực."""


def iterative_factcheck(document: str, summary: str, max_iters: int = 10) -> dict:
    if USE_MOCK:
        time.sleep(0.8)
        # Mock: giả lập kết quả mẫu
        import random
        if random.random() < 0.4:
            return {
                "verdict": "YES",
                "errors": [{
                    "erroneous_sentence": summary.split(".")[0] + "." if "." in summary else summary,
                    "error_type": "Extrinsic Error",
                    "specific": "",
                    "explanation": "[MOCK] Thông tin này không có trong văn bản gốc.",
                    "correction": "Cần kiểm tra lại với văn bản gốc.",
                }],
                "references": [],
                "raw": '{"verdict":"YES","errors":[...],"references":[]}',
            }
        return {
            "verdict": "NO",
            "errors": [],
            "references": [{"sentence_num": 1, "supporting": [document[:120] + "..."]}],
            "raw": '{"verdict":"NO","errors":[],"references":[...]}',
        }

    prompt = (
        f"VĂN BẢN GỐC:\n{document}\n\n"
        f"BẢN TÓM TẮT CẦN KIỂM TRA:\n{summary}"
    )
    last_result = {"verdict": "ERROR", "errors": [], "references": [], "raw": ""}

    for attempt in range(max_iters):
        try:
            # Ưu tiên Groq (nhanh hơn), fallback OpenAI
            if groq_client:
                raw = _call_groq(prompt, system=FACTCHECK_SYSTEM)
            elif openai_client:
                raw = _call_openai(prompt, system=FACTCHECK_SYSTEM)
            else:
                return {"verdict": "ERROR", "errors": [], "references": [],
                        "raw": "[LỖI] Không có API client nào khả dụng."}
            result = _parse_factcheck_response(raw)
            if result["verdict"] in ("YES", "NO"):
                return result
            last_result = result
        except Exception as e:
            last_result["raw"] = f"[Attempt {attempt+1}] Lỗi: {e}"
            time.sleep(1)

    return last_result


def run_comparison(document: str, summary: str) -> tuple:
    """So sánh kết quả từ 3 model: LoRA, Groq, OpenAI."""
    results = {}

    def _run_one(name, fn):
        try:
            t0 = time.time()
            r  = fn(document, summary, max_iters=3)
            r["time"] = round(time.time() - t0, 1)
            results[name] = r
        except Exception as e:
            results[name] = {"verdict": "ERROR", "errors": [], "references": [],
                             "raw": str(e), "time": 0}

    if USE_MOCK:
        time.sleep(0.5)
        for name in ["LoRA (Qwen2.5-7B)", "Groq (Llama-3.3-70B)", "GPT-4o-mini"]:
            results[name] = iterative_factcheck(document, summary, max_iters=1)
            results[name]["time"] = round(__import__("random").uniform(0.5, 2.0), 1)
    else:
        import threading
        threads = []
        if model is not None:
            threads.append(threading.Thread(
                target=_run_one, args=("LoRA (Qwen2.5-7B)", iterative_factcheck)))
        if groq_client:
            threads.append(threading.Thread(
                target=_run_one, args=("Groq (Llama-3.3-70B)", iterative_factcheck)))
        if openai_client:
            threads.append(threading.Thread(
                target=_run_one, args=("GPT-4o-mini", iterative_factcheck)))
        for t in threads: t.start()
        for t in threads: t.join()

    if not results:
        return _info_box("⚠️", "#F59E0B", "Không có model nào khả dụng để so sánh."), ""

    # Build HTML so sánh
    cmp_html = '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:16px;">'
    raw_parts = []
    for name, r in results.items():
        verdict  = r.get("verdict", "ERROR")
        n_errors = len(r.get("errors", []))
        elapsed  = r.get("time", 0)
        if verdict == "YES":
            badge_color, badge_text = "#EF4444", f"❌ Có lỗi ({n_errors})"
        elif verdict == "NO":
            badge_color, badge_text = "#22C55E", "✅ Trung thực"
        else:
            badge_color, badge_text = "#9CA3AF", "⚠️ Lỗi"
        cmp_html += f"""
        <div style="background:white;border:1px solid #E5E7EB;border-radius:12px;padding:18px;">
            <div style="font-size:13px;font-weight:700;color:#374151;margin-bottom:10px;">{name}</div>
            <div style="display:inline-block;background:{badge_color};color:white;font-size:12px;
                        font-weight:700;padding:4px 14px;border-radius:20px;margin-bottom:10px;">
                {badge_text}
            </div>
            <div style="font-size:12px;color:#9CA3AF;">⏱ {elapsed}s</div>
        </div>"""
        raw_parts.append(f"=== {name} ===\n{r.get('raw','')}")
    cmp_html += '</div>'

    raw_html = f"""<pre style="background:#1E293B;color:#E2E8F0;border-radius:10px;padding:14px;
        font-size:12px;line-height:1.6;overflow-x:auto;white-space:pre-wrap;">{chr(10).join(raw_parts)}</pre>"""
    return cmp_html, raw_html


def _translate_raw_vi(raw: str) -> str:
    """Định dạng đầu ra thô để hiển thị dễ đọc hơn."""
    if not raw:
        return "(Chưa có kết quả)"
    try:
        data  = _json.loads(re.sub(r"```(?:json)?", "", raw).strip().rstrip("`"))
        return _json.dumps(data, ensure_ascii=False, indent=2)
    except Exception:
        return raw


print('✅ Bước 4 xong — ERROR_COLORS, ETYPE_VI, logic functions đã sẵn sàng!')

✅ Bước 4 xong — ERROR_COLORS, ETYPE_VI, logic functions đã sẵn sàng!


## Bước 5: Hàm xử lý chính cho Gradio

In [13]:
# ──────────────────────────────────────────────────────────────────────
# HÀM SINH TÓM TẮT & KIỂM TRA
# ──────────────────────────────────────────────────────────────────────

def on_generate_summary(document: str):
    if not document.strip():
        return gr.update(value=""), _info_box("⚠️", "#F59E0B", "Vui lòng nhập nội dung văn bản gốc trước!")
    document = clean_doc_for_inference(document)
    summary  = generate_summary(document)
    if summary.startswith("[LỖI]"):
        return gr.update(value=""), _info_box("❌", "#EF4444", summary)
    note = _info_box(
        "✏️", "#6366F1",
        "Bản tóm tắt đã được sinh tự động. Bạn có thể chỉnh sửa trước khi nhấn <b>Kiểm chứng</b>."
    )
    return gr.update(value=summary), note


def on_check_summary(document: str, summary: str):
    if not document.strip():
        return _info_box("⚠️", "#F59E0B", "Vui lòng nhập nội dung văn bản gốc!"), "", ""
    if not summary.strip():
        return _info_box("⚠️", "#F59E0B", "Vui lòng nhập hoặc sinh bản tóm tắt trước!"), "", ""

    parsed  = iterative_factcheck(document.strip(), summary.strip(), max_iters=10)
    raw     = parsed["raw"]
    verdict = parsed["verdict"]

    mock_note = (
        '<div style="display:inline-block;background:#FFF7ED;border:1px solid #FED7AA;'
        'color:#C2410C;font-size:11px;font-weight:600;padding:2px 8px;border-radius:20px;'
        'margin-bottom:10px;">⚠️ Chế độ Mock</div><br>' if USE_MOCK else ""
    )

    if verdict == "ERROR":
        verdict_html = _info_box("❌", "#EF4444", raw)
    elif verdict == "YES":
        n = len(parsed["errors"])
        verdict_html = f"""
      {mock_note}
      <div style="background:#FEF2F2;border:2px solid #EF4444;border-radius:14px;padding:20px 24px;">
          <div style="font-size:22px;font-weight:700;color:#DC2626;margin-bottom:6px;">❌ Tóm tắt có lỗi thực tế</div>
          <div style="color:#7F1D1D;font-size:14px;">
              Phát hiện <b>{n} lỗi</b> — Bản tóm tắt không được hỗ trợ hoàn toàn bởi văn bản gốc
          </div>
      </div>"""
    else:
        verdict_html = f"""
      {mock_note}
      <div style="background:#F0FDF4;border:2px solid #22C55E;border-radius:14px;padding:20px 24px;">
          <div style="font-size:22px;font-weight:700;color:#16A34A;margin-bottom:6px;">✅ Bản tóm tắt trung thực</div>
          <div style="color:#14532D;font-size:14px;">
              Mọi câu trong bản tóm tắt đều được hỗ trợ bởi văn bản gốc
          </div>
      </div>"""

    if verdict == "YES":
        details_html = _render_errors(parsed["errors"], summary)
    elif verdict == "NO":
        details_html = _render_references(parsed["references"])
    else:
        details_html = ""

    return verdict_html, details_html, raw


def update_doc_count(text):
    n = len(text.split()) if text else 0
    color = "#EF4444" if n > 700 else "#9CA3AF"
    warn  = " ⚠️ Quá dài" if n > 700 else ""
    return f'<div style="font-size:12px;color:{color};">📊 {n} từ{warn}</div>'

def update_sum_count(text):
    n = len(text.split()) if text else 0
    return f'<div style="font-size:12px;color:#9CA3AF;">📊 {n} từ</div>'


print('✅ Bước 5 xong — Hàm xử lý đã định nghĩa xong!')

✅ Bước 5 xong — Hàm xử lý đã định nghĩa xong!


## Bước 6: Khởi chạy Gradio

In [14]:
# ──────────────────────────────────────────────────────────────────────
# VÍ DỤ MẪU (bắt buộc định nghĩa trước khi build UI)
# ──────────────────────────────────────────────────────────────────────
EXAMPLES = [
    {
        "document": (
            "Công ty VinFast vừa công bố kế hoạch xuất khẩu 100.000 xe điện sang thị trường Mỹ trong năm 2024. "
            "Chủ tịch tập đoàn Vingroup, ông Phạm Nhật Vượng, khẳng định đây là bước đi chiến lược nhằm đưa "
            "thương hiệu Việt Nam ra toàn cầu. VinFast hiện đang vận hành nhà máy sản xuất tại Hải Phòng với "
            "công suất 250.000 xe/năm. Mẫu xe VF 8 và VF 9 là hai dòng sản phẩm chủ lực được kỳ vọng chinh "
            "phục thị trường Bắc Mỹ. Giá bán dự kiến của VF 8 tại Mỹ dao động từ 40.700 đến 46.000 USD."
        ),
        "summary": (
            "VinFast lên kế hoạch xuất khẩu 100.000 xe điện sang Mỹ năm 2024. "
            "Chủ tịch Phạm Nhật Vượng xem đây là chiến lược toàn cầu hóa thương hiệu Việt. "
            "Hai mẫu VF 8 và VF 9 sẽ dẫn đầu thị trường Bắc Mỹ với giá từ 40.700 USD."
        ),
    },
    {
        "document": (
            "Đội tuyển bóng đá Việt Nam đã giành chiến thắng 2-0 trước Thái Lan trong trận chung kết "
            "AFF Cup 2022 diễn ra tại Hà Nội. Tiền đạo Nguyễn Tiến Linh ghi cả hai bàn thắng ở phút 35 "
            "và phút 67. Đây là lần thứ ba Việt Nam vô địch AFF Cup sau các năm 2008 và 2018. "
            "Huấn luyện viên trưởng Park Hang-seo bày tỏ niềm vui và cảm ơn sự cổ vũ của người hâm mộ. "
            "Khoảng 40.000 khán giả đã có mặt tại sân vận động Mỹ Đình để cổ vũ cho đội nhà."
        ),
        "summary": (
            "Việt Nam vô địch AFF Cup 2022 sau khi đánh bại Thái Lan 3-0. "
            "Tiến Linh là người hùng với cú đúp bàn thắng. "
            "Đây là lần thứ tư Việt Nam lên ngôi vô địch AFF Cup."
        ),
    },
    {
        "document": (
            "Ngân hàng Nhà nước Việt Nam vừa quyết định giữ nguyên lãi suất điều hành ở mức 4,5%/năm "
            "trong kỳ họp tháng 6/2024. Quyết định này nhằm hỗ trợ tăng trưởng kinh tế trong bối cảnh "
            "lạm phát được kiểm soát tốt ở mức 3,2%. Các chuyên gia kinh tế nhận định đây là tín hiệu "
            "tích cực cho thị trường bất động sản và doanh nghiệp vừa và nhỏ. Dự báo GDP năm 2024 của "
            "Việt Nam đạt khoảng 6,5%, cao hơn mức bình quân khu vực ASEAN."
        ),
        "summary": (
            "Ngân hàng Nhà nước giữ lãi suất điều hành 4,5%/năm để hỗ trợ tăng trưởng. "
            "Lạm phát được kiểm soát ở mức 3,2%, GDP 2024 dự báo đạt 6,5%."
        ),
    },
]

# ──────────────────────────────────────────────────────────────────────
# CSS (giữ nguyên từ đây trở xuống)
# ──────────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@300;400;500;600;700&display=swap');
* { font-family: 'Be Vietnam Pro', sans-serif !important; }
body, .gradio-container { background: #F4F6F9 !important; }

.main-header {
    background: linear-gradient(135deg, #1E40AF 0%, #7C3AED 100%);
    border-radius: 16px; padding: 28px 32px; margin-bottom: 24px;
}
.main-header h1 { font-size: 2rem !important; font-weight: 700 !important;
    margin: 0 0 6px 0 !important; color: white !important; }
.main-header p { color: rgba(255,255,255,0.8); font-size: 0.95rem; margin: 0; }

.section-label { font-size: 11px; font-weight: 700; color: #6B7280;
    text-transform: uppercase; letter-spacing: 0.8px; margin-bottom: 8px; }

textarea { border-radius: 10px !important; border: 1.5px solid #E5E7EB !important;
    font-size: 14px !important; line-height: 1.7 !important; }
textarea:focus { border-color: #6366F1 !important;
    box-shadow: 0 0 0 3px rgba(99,102,241,0.1) !important; }

.btn-generate { background: white !important; color: #4F46E5 !important;
    border: 2px solid #4F46E5 !important; border-radius: 10px !important;
    font-size: 14px !important; font-weight: 600 !important; }
.btn-generate:hover { background: #EEF2FF !important; }

.btn-check { background: linear-gradient(135deg, #4F46E5, #7C3AED) !important;
    color: white !important; border: none !important; border-radius: 10px !important;
    font-size: 15px !important; font-weight: 600 !important;
    box-shadow: 0 4px 14px rgba(79,70,229,0.3) !important; }
.btn-check:hover { transform: translateY(-1px) !important;
    box-shadow: 0 6px 20px rgba(79,70,229,0.4) !important; }

.btn-compare { background: linear-gradient(135deg, #D97706, #EA580C) !important;
    color: white !important; border: none !important; border-radius: 10px !important;
    font-size: 15px !important; font-weight: 600 !important;
    box-shadow: 0 4px 14px rgba(217,119,6,0.3) !important; }
.btn-compare:hover { transform: translateY(-1px) !important;
    box-shadow: 0 6px 20px rgba(217,119,6,0.4) !important; }

.btn-clear { background: white !important; color: #6B7280 !important;
    border: 1.5px solid #E5E7EB !important; border-radius: 10px !important; }

.gradio-accordion { border-radius: 10px !important; border: 1px solid #E5E7EB !important; }
"""

HEADER_HTML = """
<div class="main-header">
    <h1>🔍 Kiểm chứng thông tin chi tiết cho mô hình ngôn ngữ lớn</h1>
</div>"""

LEGEND_HTML = """
<div style="background:white;border:1px solid #E5E8F0;border-radius:12px;padding:14px 18px;margin-top:8px;">
    <div style="font-size:11px;font-weight:600;color:#6B7280;text-transform:uppercase;
                letter-spacing:0.8px;margin-bottom:10px;">🎨 Loại lỗi</div>
    <div style="display:flex;flex-wrap:wrap;gap:8px;">
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#FEF2F2;border:1px solid #FCA5A5;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#EF4444;display:inline-block;"></span>Lỗi vị ngữ</span>
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#FFFBEB;border:1px solid #FCD34D;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#F59E0B;display:inline-block;"></span>Lỗi thực thể</span>
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#FFF7ED;border:1px solid #FDBA74;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#F97316;display:inline-block;"></span>Lỗi hoàn cảnh</span>
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#EEF2FF;border:1px solid #A5B4FC;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#6366F1;display:inline-block;"></span>Lỗi đồng tham chiếu</span>
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#FDF2F8;border:1px solid #F9A8D4;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#EC4899;display:inline-block;"></span>Lỗi liên kết diễn ngôn</span>
        <span style="display:inline-flex;align-items:center;gap:5px;font-size:12px;
            background:#F0FDF4;border:1px solid #86EFAC;border-radius:6px;padding:4px 10px;">
            <span style="width:8px;height:8px;border-radius:50%;background:#22C55E;display:inline-block;"></span>Lỗi ngoại lai</span>
    </div>
</div>"""

EMPTY_RESULT = """
<div style="background:#F9FAFB;border:1px dashed #D1D5DB;border-radius:12px;
            padding:32px;text-align:center;color:#9CA3AF;font-size:14px;">
    Nhập văn bản gốc → <b>Sinh tóm tắt</b> → chỉnh sửa nếu cần → nhấn <b>Kiểm chứng</b>
</div>"""

# ──────────────────────────────────────────────────────────────────────
# BUILD UI (giữ nguyên hoàn toàn)
# ──────────────────────────────────────────────────────────────────────
with gr.Blocks(css=CUSTOM_CSS, title="Kiểm chứng thông tin chi tiết cho mô hình ngôn ngữ lớn") as demo:

    gr.HTML(HEADER_HTML)

    if USE_MOCK:
        gr.HTML("""
        <div style="background:#FFF7ED;border:1px solid #FED7AA;border-radius:10px;
                    padding:12px 18px;margin-bottom:16px;">
            ⚠️ <b style="color:#C2410C;">Chế độ Mock đang bật</b>
            <span style="color:#92400E;font-size:13px;">
            — Đổi <code>USE_MOCK = False</code> ở Bước 2 khi đã có model thật.
            </span>
        </div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=1):
            gr.HTML('<div class="section-label">📄 Văn bản gốc</div>')
            doc_input = gr.Textbox(
                lines=13,
                placeholder="Dán nội dung văn bản gốc vào đây...\n\nHoặc chọn ví dụ mẫu bên dưới.",
                label="", show_label=False,
            )
            doc_word_count = gr.HTML('<div style="font-size:12px;color:#9CA3AF;">0 từ</div>')

        with gr.Column(scale=1):
            gr.HTML('<div class="section-label">📝 Bản tóm tắt cần kiểm chứng</div>')
            sum_input = gr.Textbox(
                lines=13,
                placeholder="Nhấn ✨ Sinh tóm tắt để tự động điền từ văn bản gốc.\nHoặc nhập trực tiếp bản tóm tắt vào đây.",
                label="", show_label=False,
            )
            sum_word_count = gr.HTML('<div style="font-size:12px;color:#9CA3AF;">0 từ</div>')

    doc_input.change(update_doc_count, inputs=doc_input, outputs=doc_word_count)
    sum_input.change(update_sum_count, inputs=sum_input, outputs=sum_word_count)

    with gr.Row():
        gen_btn     = gr.Button("✨  Sinh tóm tắt",       scale=2, elem_classes=["btn-generate"])
        check_btn   = gr.Button("🔍  Kiểm chứng",         scale=2, variant="primary", elem_classes=["btn-check"])
        compare_btn = gr.Button("⚖️  So sánh 3 mô hình",  scale=2, elem_classes=["btn-compare"])
        clear_btn   = gr.Button("🗑️  Xóa",                scale=1, elem_classes=["btn-clear"])

    gen_note = gr.HTML("")
    gr.HTML(LEGEND_HTML)

    gr.HTML('<div style="font-size:11px;font-weight:700;color:#6B7280;text-transform:uppercase;'
            'letter-spacing:0.8px;margin:20px 0 10px;">📊 Kết quả kiểm chứng</div>')

    verdict_out = gr.HTML(value=EMPTY_RESULT)

    with gr.Tabs():
        with gr.Tab("🔍 Chi tiết lỗi / Câu nguồn"):
            details_out = gr.HTML(value="")
        with gr.Tab("⚖️ So sánh 3 mô hình"):
            gr.HTML('<div style="font-size:13px;color:#6B7280;padding:8px 0;">'
                    'Nhấn <b>⚖️ So sánh 3 mô hình</b> để chạy cả 3 model cùng lúc.</div>')
            compare_out = gr.HTML(value="")
        with gr.Tab("📄 Đầu ra thô"):
            raw_out = gr.HTML(value="")

    with gr.Accordion("💡 Ví dụ minh họa (click để thử)", open=True):
        gr.HTML('<div style="font-size:12px;color:#6B7280;margin-bottom:10px;">'
                'Click vào một hàng để tải tài liệu và tóm tắt tương ứng.</div>')
        gr.Examples(
            examples=[[ex["document"], ex["summary"]] for ex in EXAMPLES],
            inputs=[doc_input, sum_input],
            label="",
            examples_per_page=5,
        )
        gr.HTML('<div style="font-size:11px;color:#9CA3AF;margin-top:6px;">'
                '🔴 Lỗi vị ngữ &nbsp;🟡 Lỗi thực thể &nbsp;🟠 Lỗi hoàn cảnh &nbsp;'
                '🟣 Lỗi đồng tham chiếu &nbsp;🔵 Lỗi liên kết diễn ngôn &nbsp;🟢 Lỗi ngoại lai</div>')

    with gr.Accordion("ℹ️ Về hệ thống & quy trình xây dựng", open=False):
        gr.Markdown("""
## Kiểm chứng thông tin chi tiết cho mô hình ngôn ngữ lớn

**Hệ thống** kiểm tra tính chính xác của tóm tắt do LLM sinh ra.

### Pipeline nghiên cứu (7 bước):
```
Bước 1: dataset_vnexpress.py          → Lọc bài báo VnExpress (150–700 từ)
Bước 2: Upload lên Google Drive
Bước 3: summary_gen_pipeline.ipynb   → Llama-3.3-70B sinh tóm tắt
Bước 4: eval_and_reference_gen.ipynb → 3-model voting xác nhận tóm tắt
Bước 5: structured_dataset_gen.ipynb → GPT-4o-mini tạo lỗi nhân tạo (6 loại)
Bước 6: prepare_dataset_pipeline.ipynb → Xuất JSONL train/valid/test
Bước 7: Finetune_Colab.ipynb → QLoRA tinh chỉnh Qwen2.5-7B
```

### Flow demo này:
```
Tài liệu → [✨ Sinh tóm tắt] → Tóm tắt (sửa được) → [🔍 Kiểm chứng] → Kết quả
```

### Các loại lỗi:
| | Loại | Ví dụ |
|---|---|---|
| 🔴 | Lỗi vị ngữ | "tăng" → "giảm" |
| 🟡 | Lỗi thực thể | bỏ mất "Xiaomi" khỏi danh sách |
| 🟠 | Lỗi hoàn cảnh | ngày 15/3 → 20/5 |
| 🟣 | Lỗi đồng tham chiếu | "ông ấy" → "bà ấy" |
| 🔵 | Lỗi liên kết diễn ngôn | đảo ngược nhân quả |
| 🟢 | Lỗi ngoại lai | thêm thông tin không có trong tài liệu |
        """)

    # ── Event handlers ──────────────────────────────────────────────

    def handle_generate(doc):
        return on_generate_summary(doc)

    def handle_check(doc, summ):
        v, d, r = on_check_summary(doc, summ)
        raw_html = f'''<div style="margin-bottom:16px;">
            <pre style="background:#1E293B;color:#E2E8F0;border-radius:10px;padding:14px;
                        font-size:12px;line-height:1.6;overflow-x:auto;white-space:pre-wrap;">{_translate_raw_vi(r)}</pre>
        </div>'''
        return v, d, raw_html

    def handle_compare(doc, summ):
        if not doc.strip() or not summ.strip():
            warn = _info_box("⚠️", "#F59E0B", "Vui lòng nhập đủ tài liệu và tóm tắt!")
            return gr.update(), warn, ""
        cmp_html, raw_html = run_comparison(doc.strip(), summ.strip())
        return EMPTY_RESULT, cmp_html, raw_html

    def handle_clear():
        return "", "", "", EMPTY_RESULT, "", ""

    gen_btn.click(
        fn=handle_generate,
        inputs=[doc_input],
        outputs=[sum_input, gen_note],
    )
    check_btn.click(
        fn=handle_check,
        inputs=[doc_input, sum_input],
        outputs=[verdict_out, details_out, raw_out],
        api_name="check",
        show_progress="minimal",
    )
    compare_btn.click(
        fn=handle_compare,
        inputs=[doc_input, sum_input],
        outputs=[verdict_out, compare_out, raw_out],
        api_name="compare",
        show_progress="minimal",
    )
    clear_btn.click(
        fn=handle_clear,
        inputs=[],
        outputs=[doc_input, sum_input, gen_note, verdict_out, details_out, raw_out],
    )

# ──────────────────────────────────────────────────────────────────────
# LAUNCH
# ──────────────────────────────────────────────────────────────────────
print('🚀 Đang khởi động Gradio...')
demo.launch(share=True, debug=False, show_error=True, quiet=False, max_threads=4)
print('✅ Gradio đang chạy! Click vào public link ở trên.')

/tmp/ipykernel_31495/1627685140.py:137: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Kiểm chứng thông tin chi tiết cho mô hình ngôn ngữ lớn") as demo:


🚀 Đang khởi động Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a54ffb8349602d2041.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Gradio đang chạy! Click vào public link ở trên.
